# Categorized error analysis of the M3 pipeline (dev)

**Question.** Where do the 360 dev errors of the M3 pipeline (BERT retriever + T5 program generator,
Notebooks 10–11) come from, and which error type should the first M4 improvement target?

**Why it matters.** M4 requires a categorized error analysis before any improvement iteration or the
single test run. Improving blindly would risk tuning on noise.

**Protocol.** Dev only (883 records), using the saved Notebook 11 prediction files. The test split is
not loaded. Dev labels (`exe_ans`, `program`, `gold_inds`) are read here for diagnosis, which is the
dev split's purpose; nothing here is a model input.

In [1]:
from __future__ import annotations

import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

from finevidence.data import load_finqa_split, parse_finqa_program
from finevidence.data.acquisition import PINNED_SHA256, file_sha256
from finevidence.evaluation import CATEGORIES, categorize_error

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RUN = "generator-t5-base-dev-v1"

assert file_sha256(RAW_DIR / "dev.json") == PINNED_SHA256["dev"]
dev = {record["id"]: record for record in load_finqa_split(RAW_DIR / "dev.json")}


def read_predictions(condition: str) -> dict[str, dict]:
    path = PROCESSED_DIR / f"{RUN}_dev_{condition}_predictions.jsonl"
    rows = [json.loads(line) for line in path.read_text().splitlines()]
    return {row["record_id"]: row for row in rows}


retrieved = read_predictions("retrieved")
oracle = read_predictions("gold_facts_oracle")
assert set(retrieved) == set(oracle) == set(dev)
correct = sum(row["execution_correct"] for row in retrieved.values())
print(f"dev records: {len(dev)}, retrieved correct: {correct}, errors: {len(dev) - correct}")

dev records: 883, retrieved correct: 523, errors: 360


## 1. Categories

`finevidence.evaluation.categorize_error` gives each error exactly one primary category, checked in a
fixed priority order: invalid program, retrieval miss (a gold fact was not among the 5 input facts),
scale error (off by a power of ten), sign error, near miss (within 1%), operation error (different
operation sequence from the gold program), operand error (same operations, different arguments), other.
These are deterministic proxies, not causal labels: a scale error, for example, can still involve a
wrong operand.

In [2]:
YEAR = re.compile(r"\b(?:19|20)\d{2}\b")
rows = []
for record_id, prediction in retrieved.items():
    if prediction["execution_correct"]:
        continue
    record = dev[record_id]
    qa = record["qa"]
    category = categorize_error(
        predicted_program=prediction["prediction"] if prediction["valid"] else None,
        predicted_value=prediction["executed_value"] if prediction["valid"] else None,
        reference_value=qa["exe_ans"],
        gold_program=qa["program"],
        gold_keys=set(qa["gold_inds"]),
        input_keys=set(prediction["fact_keys"]),
        table=record["table"],
    )
    steps = len(parse_finqa_program(qa["program"]).steps)
    rows.append(
        {
            "id": record_id,
            "category": category.primary,
            "operand_detail": category.operand_detail,
            "gold_steps": "1" if steps == 1 else "2" if steps == 2 else "3+",
            "gold_facts": "1" if len(qa["gold_inds"]) == 1 else "2+",
            "oracle_correct": oracle[record_id]["execution_correct"],
            "question_years": len(set(YEAR.findall(qa["question"]))),
        }
    )
errors = pd.DataFrame(rows)
counts = errors["category"].value_counts().reindex(CATEGORIES, fill_value=0)
summary = pd.DataFrame(
    {
        "errors": counts,
        "share_of_errors": (counts / len(errors)).round(3),
        "share_of_dev": (counts / len(dev)).round(3),
    }
)
print(f"errors categorized: {len(errors)}")
summary

errors categorized: 360


,errors,share_of_errors,share_of_dev
category,,,
invalid_program,7,0.019,0.008
retrieval_miss,81,0.225,0.092
scale_error,14,0.039,0.016
sign_error,14,0.039,0.016
near_miss,13,0.036,0.015
operation_error,123,0.342,0.139
operand_error,108,0.300,0.122
other,0,0.000,0.000


## 2. Categories against program length and the gold-facts oracle

`oracle_correct` says whether the same generator answered correctly when given only the gold facts.
For retrieval misses it estimates how many errors better retrieval alone could fix.

In [3]:
pd.crosstab(errors["category"], errors["gold_steps"], margins=True).reindex(
    list(CATEGORIES) + ["All"], fill_value=0
)

gold_steps,1,2,3+,All
category,,,,
invalid_program,6,0,1,7
retrieval_miss,32,34,15,81
scale_error,6,5,3,14
sign_error,8,4,2,14
near_miss,6,6,1,13
operation_error,51,47,25,123
operand_error,71,32,5,108
other,0,0,0,0
All,180,128,52,360


In [4]:
pd.crosstab(errors["category"], errors["oracle_correct"], margins=True).reindex(
    list(CATEGORIES) + ["All"], fill_value=0
)

oracle_correct,False,True,All
category,,,
invalid_program,5,2,7
retrieval_miss,64,17,81
scale_error,14,0,14
sign_error,10,4,14
near_miss,10,3,13
operation_error,107,16,123
operand_error,66,42,108
other,0,0,0
All,276,84,360


In [5]:
operand = errors[errors["category"] == "operand_error"]
print("operand errors:", len(operand))
operand["operand_detail"].value_counts()

operand errors: 108


operand_detail
other_value                                  73
same_literals_different_order_or_constant    19
same_table_row_as_gold                       16
Name: count, dtype: int64

In [6]:
print("questions naming 2+ distinct years, share by category:")
errors.assign(multi_year=errors["question_years"] >= 2).groupby("category")["multi_year"].agg(
    ["sum", "count", "mean"]
).reindex(CATEGORIES).round(3)

questions naming 2+ distinct years, share by category:


,sum,count,mean
category,,,
invalid_program,2.0,7.0,0.286
retrieval_miss,19.0,81.0,0.235
scale_error,5.0,14.0,0.357
sign_error,9.0,14.0,0.643
near_miss,5.0,13.0,0.385
operation_error,41.0,123.0,0.333
operand_error,38.0,108.0,0.352
other,NaN,NaN,NaN


### Operation errors by shape

How the predicted operation sequence differs from the gold one: a strict prefix (the generator stopped
early), the gold sequence plus extra steps, or a different sequence with fewer, more or the same number
of steps.

In [7]:
def operation_shape(gold_program: str, predicted_program: str) -> str:
    gold = [step.operation for step in parse_finqa_program(gold_program).steps]
    predicted = [step.operation for step in parse_finqa_program(predicted_program).steps]
    if len(predicted) < len(gold) and predicted == gold[: len(predicted)]:
        return "stopped early (prefix of gold)"
    if len(predicted) > len(gold) and predicted[: len(gold)] == gold:
        return "gold plus extra steps"
    if len(predicted) < len(gold):
        return "fewer steps, different operations"
    if len(predicted) > len(gold):
        return "more steps, different operations"
    return "same length, different operations"


operation_errors = errors[errors["category"] == "operation_error"]
Counter(
    operation_shape(dev[i]["qa"]["program"], retrieved[i]["prediction"])
    for i in operation_errors["id"]
).most_common()

[('fewer steps, different operations', 39),
 ('same length, different operations', 30),
 ('more steps, different operations', 28),
 ('gold plus extra steps', 13),
 ('stopped early (prefix of gold)', 13)]

## 3. Real examples per category

Two examples per category (the first two in file order, no cherry-picking), showing the question, the
gold program and value, and the predicted program and value.

In [8]:
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
examples = []
for category in CATEGORIES:
    for record_id in errors.loc[errors["category"] == category, "id"].head(2):
        qa = dev[record_id]["qa"]
        prediction = retrieved[record_id]
        examples.append(
            {
                "category": category,
                "id": record_id,
                "question": qa["question"],
                "gold_program": qa["program"],
                "exe_ans": qa["exe_ans"],
                "predicted": prediction["prediction"],
                "predicted_value": prediction["executed_value"],
            }
        )
pd.DataFrame(examples)

,category,id,question,gold_program,exe_ans,predicted,predicted_value
0,invalid_program,AMAT/2014/page_35.pdf-1,what percentage of warehouse locations are in north america?,"divide(16, 75)",2.133300e-01,"divide(1676000 145000, 1358000164000)",NaN
1,invalid_program,GS/2014/page_136.pdf-1,"in millions in 2014 , 2013 , and 2012 , what was the greatest amou...","table_max(hedged borrowings and bank deposits, none)",6.999000e+03,"table_max(edged borrowings and bank deposits, none)",NaN
2,retrieval_miss,DVN/2007/page_58.pdf-2,what percentage of the total oil and gas mmboe comes from canada?,"divide(60, 243), multiply(#0, const_100)",2.469136e+01,"divide(60, 66)",9.090900e-01
3,retrieval_miss,DVN/2014/page_85.pdf-1,"what is the average amortization amount , in millions , from 2015-...","multiply(45, const_4), add(#0, 44), divide(#1, const_5)",4.480000e+01,"add(45, 44), add(#0, 45), add(#1, const_3), divide(#2, const_2)",6.850000e+01
4,scale_error,ZBH/2003/page_40.pdf-1,what was the difference in operating profit for the americas as a ...,"subtract(51.2, 47.4)",3.800000e+00,"subtract(51.2%, 47.4%)",3.800000e-02
5,scale_error,AES/2017/page_157.pdf-2,"as of december 31 , 2017 , assuming an average price per share of ...","multiply(246, 12.12)",2.981520e+03,"multiply(246, 12.12), divide(#0, const_1000000)",2.980000e-03
6,sign_error,PM/2018/page_24.pdf-2,what was the difference in percentage return for pmi common stock ...,"subtract(96.50, const_100), divide(#0, const_100), subtract(150.30...",5.380000e-01,"subtract(96.50, const_100), divide(#0, const_100), subtract(150.30...",-5.380000e-01
7,sign_error,ETR/2017/page_114.pdf-1,what is the difference in payments between entergy arkansas and en...,"subtract(6, 2)",4.000000e+00,"subtract(2, 6)",-4.000000e+00
8,near_miss,AMT/2010/page_34.pdf-2,on february 112011 what was the market capitalization,"multiply(397612895, 56.73)",2.255658e+10,"multiply(397611895, 56.73)",2.255652e+10
9,near_miss,LLY/2018/page_63.pdf-3,what was the percent of the finished products to the total inventory,"divide(988.1, 4111.8)",2.403100e-01,"divide(988.1, 4112.8)",2.402500e-01


In [9]:
out = PROCESSED_DIR / f"{RUN}_dev_error_categories.csv"
errors.to_csv(out, index=False)
print(out.relative_to(PROJECT_ROOT), len(errors), "rows")

data/processed/generator-t5-base-dev-v1_dev_error_categories.csv 360 rows


## 4. Interpretation (from the outputs above)

**Reasoning, not retrieval, is the larger error source.** Of 360 errors, only 81 (22.5%) are retrieval
misses. 272 errors (75.6%) occur although every gold fact was among the 5 input facts: operation errors
123, operand errors 108, scale 14, sign 14, near misses 13. Invalid programs are rare (7).

**Better retrieval alone has a small ceiling.** The gold-facts oracle fixes only 17 of the 81 retrieval
misses; the other 64 stay wrong even with perfect evidence. Perfect retrieval would therefore recover
about 17 records (about 1.9 points of dev accuracy) on these errors.

**Distractors confuse operand selection.** The oracle fixes 42 operand errors and 16 operation errors
whose gold facts were already retrieved. Of the oracle's 84 gains, 65 come from records with complete
retrieval, so removing the distractor facts helped more than adding missing facts did.

**Multi-step composition fails as structure, not arithmetic.** 72 of 123 operation errors have 2+ step
gold programs. The predicted sequence is a prefix of the gold one in 13 cases and gold plus extra steps
in 13; the remaining 97 use a different sequence.

**Small, recognisable groups:**
- Sign errors: 9 of 14 questions name two or more years. The subtraction direction is reversed
  (for example `subtract(2, 6)` for `subtract(6, 2)`).
- Near misses: the examples are single-digit copy errors (`397612895` copied as `397611895`,
  `4111.8` as `4112.8`).
- Scale errors: the examples mix units or percent conventions, such as `subtract(51.2%, 47.4%)` where
  the gold program uses percentage points.
- Operand errors: 16 use a value from the same table row as a gold value (a period or column
  confusion proxy); 19 use the gold literals in a different order or with a different constant,
  such as `divide(1189, 36197)` for `divide(36197, 1189)`.

**Not yet covered: ambiguity.** Rules cannot detect an ambiguous question or a debatable gold label.
Some examples suggest both: `ETR/2011/page_341.pdf-3` asks for a "net change" whose gold program is
`add(18.9, 0.3)`. A manual review of a fixed sample is needed before any error count is attributed
to ambiguity.

**Limitations.** One primary category per error, in a fixed priority order; categories are proxies,
not causes. Dev only, one checkpoint selected on dev, one seed.

**Candidate first improvements, in order of expected yield (hypotheses, not results):**
1. Make the generator robust to distractor facts: train with retrieved-style hard distractors, or pass
   fewer, reranked facts. Upper bound from the oracle gains on fully retrieved records: 65 records.
2. Reduce copy errors and swapped operands: constrain literals to numbers present in the input facts.
3. Retrieval improvements, with a measured ceiling of about 17 records on these errors.


## 5. Manual review of a fixed sample: ambiguity and label quality

Rules cannot tell an ambiguous question or a wrong gold label from a model error. A fixed random sample of
40 errors (`random_state=42`) was reviewed by hand against the question, the gold facts, the gold program
and the prediction. Each error got one label, stored with a short note in `docs/m4-ambiguity-review.csv`:

- `clear_model_error`: the question and gold are clear, and the prediction is wrong.
- `ambiguous_question`: the question or its context supports more than one reasonable reading.
- `debatable_gold`: the gold program looks wrong or contradicts the question or display answer.
- `convention_mismatch`: the prediction equals the gold value under another percent or unit convention.

The review was done by one reviewer (Claude, the coding assistant, at the user's request) and is a
judgement, not a measurement. The cell below re-draws the sample and checks it matches the labelled ids.

In [10]:
from scipy.stats import binomtest

REVIEW_PATH = PROJECT_ROOT / "docs" / "m4-ambiguity-review.csv"
review = pd.read_csv(REVIEW_PATH)
sample_ids = sorted(errors.sample(n=40, random_state=42)["id"])
assert sorted(review["id"]) == sample_ids, "review file does not match the seeded sample"
reviewed = review.merge(errors[["id", "category"]], on="id")
label_counts = reviewed["label"].value_counts()
interval = binomtest(
    int((reviewed["label"] != "clear_model_error").sum()), len(reviewed)
).proportion_ci(method="wilson")
low, high = interval.low, interval.high
print(
    f"not a clear model error: {(reviewed['label'] != 'clear_model_error').mean():.2f} "
    f"(Wilson 95% CI {low:.2f}-{high:.2f}, n={len(reviewed)})"
)
label_counts

not a clear model error: 0.35 (Wilson 95% CI 0.22-0.50, n=40)


label
clear_model_error      26
debatable_gold          7
ambiguous_question      4
convention_mismatch     3
Name: count, dtype: int64

In [11]:
pd.crosstab(reviewed["category"], reviewed["label"], margins=True)

label,ambiguous_question,clear_model_error,convention_mismatch,debatable_gold,All
category,,,,,
near_miss,0,1,0,1,2
operand_error,1,7,0,2,10
operation_error,2,12,0,2,16
retrieval_miss,0,5,2,1,8
scale_error,0,0,1,1,2
sign_error,1,1,0,0,2
All,4,26,3,7,40


### Interpretation of the review

- **26 of 40 sampled errors (65%) are clear model errors.** 14 of 40 (35%, Wilson 95% CI 22–50%) are
  not: 7 debatable gold labels, 4 ambiguous questions and 3 convention mismatches.
- **Some "errors" are arguably right.** In 2 debatable-gold cases the prediction follows the question
  and the gold does not: `BDX/2018/page_82.pdf-2` (the ratio is inverted in the gold) and
  `ETR/2017/page_26.pdf-1` (the gold uses the later year as the base of a percent change).
- **Convention mismatches** are percent-versus-ratio differences (`0.165` against `16.5`); the official
  metric counts them wrong.
- **Copy errors recur among the clear errors**: 5 of 26 use a number that differs from the evidence by
  one or two digits or is a year (`14669` as `14699`, `0.40` as `0.49`, `187103` as `18703`, `2010`
  for `210`, `415` for `431`).

**What this means.** A sizeable part of the measured error is benchmark noise that no model change can
fix under the official metric. The 22–50% interval is wide (n = 40), so it should not be scaled up to an
exact count of the 360 errors. The gold labels and the metric stay unchanged; this is reported as an
evaluation caveat.

**Decision.** The improvement priority stands: distractor robustness first, then constraining program
literals to numbers present in the input facts (which targets the copy errors). The ambiguity category
of the M4 error analysis is now covered by this sample.